<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_grounding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Grounding

Grounding turns a natural-language phrase into bounding boxes. GarmentIQ uses it to give
SAM 1 and SAM 2 text-prompted segmentation, since neither model contains a text encoder.

This tutorial shows how to load Grounding DINO, convert a phrase such as "a shirt" into
boxes, tune the detection thresholds, and pass the result into segmentation. Because the
boxes are model agnostic, grounding lives in its own module and works with any
prompt-driven model.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Ground a phrase into boxes](#ground)
3. [Tune the thresholds](#thresholds)
4. [Use grounding with segmentation](#segmentation)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the test image, Grounding DINO, and a SAM model. On Colab
you can keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch
from PIL import Image

import garmentiq as giq
from garmentiq.grounding import (
    load_grounding_model,
    load_grounding_processor,
    ground_text_to_boxes,
)
from garmentiq.segmentation.model_definition.sam import (
    SamModel,
    load_sam_config,
    load_sam_processor,
)

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the test image and the model weights

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_1.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_1.jpg

# Grounding DINO. The whole directory is needed because the processor bundles a tokenizer.
!mkdir -p ./models/gdino
for _f in [
    "config.json", "preprocessor_config.json", "tokenizer.json",
    "tokenizer_config.json", "special_tokens_map.json", "vocab.txt",
    "model.safetensors",
]:
    !wget -q -O ./models/gdino/{_f} https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/main/{_f}

# SAM 1, base, to consume the boxes at the end of the tutorial
!mkdir -p ./models/sam_b
!wget -q -O ./models/sam_b/model.safetensors \
    https://huggingface.co/facebook/sam-vit-base/resolve/main/model.safetensors

print("Downloads finished.")

<a name="ground"></a>
## Ground a phrase into boxes

Grounding DINO has a dedicated loader. It ties several decoder heads to one shared set of
weights, and only `from_pretrained` performs that tying correctly, so loading it through
the generic model loader would leave those tensors randomly initialized.

`ground_text_to_boxes` returns boxes as `[x_min, y_min, x_max, y_max]` in pixel
coordinates, ordered by descending confidence.

In [ ]:
grounder = load_grounding_model("./models/gdino", device=device)
grounding_processor = load_grounding_processor("./models/gdino")

image = Image.open("./test_image/cloth_1.jpg").convert("RGB")
print("image size (w, h):", image.size)

boxes = ground_text_to_boxes(
    model=grounder,
    processor=grounding_processor,
    image=image,
    text="a shirt",
    box_threshold=0.3,
    text_threshold=0.3,
    device=device,
)

for i, b in enumerate(boxes):
    print(f"box {i}: {[round(v) for v in b]}")

The phrase is free text. Grounding DINO expects lowercase phrases ending
in a period, and GarmentIQ normalizes your input to that convention automatically.

In [ ]:
for phrase in ["a shirt", "sleeve", "the neckline"]:
    try:
        found = ground_text_to_boxes(
            model=grounder,
            processor=grounding_processor,
            image=image,
            text=phrase,
            max_boxes=1,
            device=device,
        )
        print(f"{phrase:<15} -> {[round(v) for v in found[0]]}")
    except ValueError as e:
        print(f"{phrase:<15} -> no match ({e})")

<a name="thresholds"></a>
## Tune the thresholds

`box_threshold` is the minimum detection confidence and `text_threshold` the minimum
text-matching score. Raising them returns fewer but more certain boxes. `max_boxes` keeps
only the highest-scoring few, which is usually what you want before handing a box to SAM.

If nothing clears the thresholds, a `ValueError` is raised rather than an empty list, so a
failed grounding cannot quietly become an empty prompt.

In [ ]:
for t in [0.1, 0.3, 0.5]:
    try:
        found = ground_text_to_boxes(
            model=grounder,
            processor=grounding_processor,
            image=image,
            text="a shirt",
            box_threshold=t,
            text_threshold=t,
            device=device,
        )
        print(f"threshold={t} -> {len(found)} box(es)")
    except ValueError as e:
        print(f"threshold={t} -> {e}")

<a name="segmentation"></a>
## Use grounding with segmentation

You rarely need to call `ground_text_to_boxes` yourself. Pass a `text` prompt plus a
grounding model to `segmentation.extract` and GarmentIQ runs both steps for you.

In [ ]:
sam = giq.segmentation.load_model(
    model_class=SamModel,
    model_path="./models/sam_b/model.safetensors",
    model_args={"config": load_sam_config("sam-vit-b")},
    device=device,
)

original_img, mask = giq.segmentation.extract(
    model=sam,
    image_path="./test_image/cloth_1.jpg",
    processor=load_sam_processor("sam-vit-b"),
    prompt={"text": "a shirt"},
    grounding_model=grounder,
    grounding_processor=grounding_processor,
    grounding_args={"box_threshold": 0.3, "text_threshold": 0.3, "max_boxes": 1},
    device=device,
)

giq.segmentation.plot(image_np=mask, figsize=(3, 3))

Grounding is only needed for SAM 1 and SAM 2. SAM 3 understands text
natively, so it takes a `text` prompt with no grounding model at all. See the
[segmentation tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_segmentation.ipynb).